# 00 - Environment check
Checks the Python/torch/CUDA versions, that the RTX 5070 (compute capability 12.0, `sm_120`) is
supported, bf16 matmul throughput, and run-to-run determinism. Writes `results/env.json`.

In [ ]:
# parameters
MATMUL_N = 4096
REPEATS = 50

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.utils.io import env_info, write_json
info = env_info()
for k, v in info.items():
    print(f"{k:22s} {v}")
assert torch.cuda.is_available(), "CUDA not available"
assert tuple(info["capability"]) == (12, 0) and info["sm_120_in_arch_list"]

In [ ]:
a = torch.randn(MATMUL_N, MATMUL_N, device="cuda", dtype=torch.bfloat16)
for _ in range(3):
    a @ a
torch.cuda.synchronize()
start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(REPEATS):
    a @ a
end.record(); torch.cuda.synchronize()
sec = start.elapsed_time(end) / 1e3
info["bf16_matmul_tflops"] = round(REPEATS * 2 * MATMUL_N**3 / sec / 1e12, 1)
print("bf16 TFLOPS:", info["bf16_matmul_tflops"])

In [ ]:
# determinism: two identically seeded short training runs give identical weights
from glwm2.utils.config import RunConfig
from glwm2.train.trainer import train
def tiny_run():
    cfg = RunConfig(task="addition", iters=30, batch_size=64, width=32, t_train=8, eval_n=0, warmup=5)
    return train(cfg, save=False, deterministic=True)["model"]
m1, m2 = tiny_run(), tiny_run()
same = all(torch.equal(p, q) for p, q in zip(m1.state_dict().values(), m2.state_dict().values()))
info["deterministic_training"] = bool(same)
print("deterministic:", same)

In [ ]:
path = write_json(RESULTS_DIR / "env.json", info); WRITTEN.append(path)

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))